# Results  

This notebook creates files *candidates_all.fits*, and *candidates_best.fits*. The first contains a table with all candidates found at all plate pairs and sequences. This table contains the end result of the search algorithm. The second table contains the subset of candidates that, after visual inspection, look more promising as being real, and not artifacts that mimic stars in their integrated properties.

Results from the pipeline runs are stored in subdirectory *plateanalysis/footprints/results/*. Note that these HTML files include *all* results **before** visual vetting took place. Results *after* visual vetting are included in notebook *results_analysis.ipynb*.

Each HTML file contains the result of a pipeline run over a *sequence* of plates. Sequences of plates are defined in file *settings.py*, and created based on the output of notebook *footprints.ipynb*.

HTML files contain just the very final product for each sequence; intermediate results are not stored in GitHub due to their policy of minimizing the storage of files that are either temporary, or potentially large, such as FITS binary tables and such. Intermediate results can be seen only by running the code locally over the appropriate data sets.  

#### Note

Due to sheer size, only one of the plate sequences I worked with has its pipeline output uploaded to the repo. Also, GitHub is unable to display the html due to its size. Download the file and use your browser locally to display the file.  

Alternatively, you can download the result files from Google Drive: https://drive.google.com/drive/folders/164xc4faMDbPt84ZileJZoHzK94kvG8Gm


Below is a summary of findings to date. Numbers refer to the plate ID code in the APPLAUSE archive.

- Data from the **1m-Spiegelteleskop** and the **Doppel-Reflektor** telescopes: still to be reviewed in light of the upgraded software.

- Data from the **Grosser Schmidt-Spiegel** telescope yeld promising transient candidates.




### Notes

APPLAUSE scans were performed on the original plates, not copies.

## Processing stats

Here, we build a summary of number of detections/transients for each plate pair, plus plate metadata information. The goal is to provide a bird's view of the coverage of the project to date, as well as a complete list of candidates in a single table.

In [1]:
import os
import warnings
import shutil
from datetime import datetime

import numpy as np

from astropy.table import Table, vstack, hstack
from astropy.time import Time
from astropy.utils.exceptions import AstropyWarning

import pandas as pd
from erfa import ErfaWarning

from library import get_earth_shadow
import settings
from settings import CATALOG, RESULTS, get_parameters, fname, tel_suffix, sequences

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [2]:
warnings.simplefilter('ignore', category=AstropyWarning)
warnings.simplefilter('ignore', category=ErfaWarning)
warnings.simplefilter('ignore', category=FutureWarning)

In [3]:
table_header = {
    'summary': {
        'names': ['seq', 'plate 1', 'ut_mid_1', 'exptime_1', 'es_1','plate 2', 'ut_mid_2', 
                  'exptime_2', 'es_2', 'total sources', 'matched', 'non-matched', 'candidates'],
        'dtypes': ['S1', 'S1', 'S1', 'i4', 'f4', 'S1', 'S1', 'i4', 'f4', 'i4', 'i4', 'i4', 'i4'],
    },
    'metadata': {
        'names': ['seq', 'ut_mid', 'exptime', 'es', 'plate_id_next', 'ut_mid_next', 'exptime_next', 'es_next'],
        'dtypes': ['S1', 'S1', 'i4', 'f4', 'i4', 'S1', 'i4', 'f4'],
    },
}

In [4]:
def make_table(table_type):
    table = Table(names=table_header[table_type]['names'],
                  dtype=table_header[table_type]['dtypes'])
    return table

In [5]:
# use Pandas to display table so all rows can be displayed at once. Displaying long
# Astropy tables directly causes only the first and last few rows to be displayed. 

def display_with_pandas(table):
    df = table.to_pandas()

    # avoid display of unicode reminder in every string field
    df['seq'] = df['seq'].apply(lambda x: x.decode('utf-8'))
    df['plate 1'] = df['plate 1'].apply(lambda x: x.decode('utf-8'))
    df['plate 2'] = df['plate 2'].apply(lambda x: x.decode('utf-8'))
    df['ut_mid_1'] = df['ut_mid_1'].apply(lambda x: x.decode('utf-8'))
    df['ut_mid_2'] = df['ut_mid_2'].apply(lambda x: x.decode('utf-8'))

    # style with CSS properties for table headers (th) and data cells (td)
    th_props = [('font-size', '12px'), ('text-align', 'right'), ('font-weight', 'bold')]
    td_props = [('font-size', '11px'), ('text-align', 'right')]
    styles = [dict(selector="th", props=th_props),dict(selector="td", props=td_props)]
    styled_df = df.style.set_table_styles(styles)    
    
    format_mapping = {
        'es_1': '{:.4}'.format,
        'es_2': '{:.4}'.format,
    }
    styled_df = styled_df.format(format_mapping)    

    display(styled_df)

In [6]:
# get metadata for a given plate. Metadata is taken from original input table from applause.

def get_metadata(plate, catalog_table):
    
    mask = catalog_table['plate_id'] == plate
    t = catalog_table[mask]
    
    long = t['site_longitude'][0]
    lat  = t['site_latitude'][0]
    ra  = t['ra_icrs'][0]
    dec = t['dec_icrs'][0]
    time_event = Time(t['ut_mid'][0])

    es, _ = get_earth_shadow(ra, dec, time_event, longitude=long, latitude=lat)
    
    return t['ut_mid'][0], t['exptime'][0], es

In [7]:
def add_metadata(metadata, table):
    
    # create temporary table to contain metadata
    meta_table = make_table('metadata')
    
    for row_index in range(len(table)):
        plate_id = table['plate_id_1'][row_index]

        seq           = metadata[plate_id]['seq']    
        plate_id_next = metadata[plate_id]['plate_id_next']    
        ut_mid        = metadata[plate_id]['ut_mid']    
        exptime       = metadata[plate_id]['exptime']    
        es            = metadata[plate_id]['es']    
        ut_mid_next   = metadata[plate_id]['ut_mid_next']    
        exptime_next  = metadata[plate_id]['exptime_next']    
        es_next       = metadata[plate_id]['es_next']    
    
        meta_table.add_row([seq, ut_mid, exptime, es, plate_id_next, ut_mid_next, exptime_next, es_next])
  
    # join with main table
    result = hstack([meta_table, table])
    
    return result    

In [8]:
# metadata comes from original input table from applause
catalog_table = Table.read(CATALOG, format='ascii.csv')

# summary table
result = make_table('summary')

# list of tables to collect all candidate events
candidates_list = []

# collect some metadata here
metadata = {}

In [9]:
# 'sequences' is a dict defined in file settings.py

for seq_key in list(sequences.keys()):
    seq = sequences[seq_key]

    for i in range(len((seq)) - 1):
        try:
            plate_id = seq[i]
            next_plate_id = seq[i+1]
            
            # metadata
            ut_mid_1, exptime_1, es_1 = get_metadata(plate_id, catalog_table)
            ut_mid_2, exptime_2, es_2 = get_metadata(next_plate_id, catalog_table)
            
            metadata[plate_id] = {
                'seq': seq_key,
                'ut_mid': ut_mid_1,
                'exptime': exptime_1,
                'es': es_1,
                'plate_id_next': next_plate_id,
                'ut_mid_next': ut_mid_2,
                'exptime_next': exptime_2,
                'es_next': es_2,
            }
            
            plate_id_str = str(plate_id)
            next_plate_id_str = str(next_plate_id)

            key = plate_id_str + ',' + next_plate_id_str
            par = get_parameters(key)

            # row count on each relevant product table
            table_sources = Table.read(fname(par['table1']), format='ascii.csv')
            mask = table_sources['scan_id'] == table_sources['scan_id'][0]
            table_sources = table_sources[mask]
            sources = len(table_sources)

            table_matched = Table.read(fname(par['table_matched']), format='fits')
            matched = len(table_matched)

            table_non_matched = Table.read(fname(par['table_non_matched']), format='fits')
            non_matched = len(table_non_matched)

            table_candidates = Table.read(fname(par['table_candidates']), format='fits')
            candidates = len(table_candidates)
            
            # add this candidates table to final product. Add seq_key as identifier
            candidates_list.append((seq_key, table_candidates))
            
            result.add_row([seq_key, plate_id_str, ut_mid_1, exptime_1, es_1,
                            next_plate_id_str, ut_mid_2, exptime_2, es_2,
                            sources, matched, non_matched, candidates])
        except (KeyError, FileNotFoundError):
            break

### Summary table derived from pipeline run

Table columns:

- **seq**: sequence ID generated by notebook *footprints.ipynb*
- **plate ...**: plate IDs defining the pair of plates
- **ut_mid ...**: UT mid for each plate
- **exptime ...**: EXPTIME for each plate
- **es ...**: Earth's shadow angle (deg) for each plate
- **total sources**: total number of sources in the APPLAUSE table of the first plate, for the *_x* scan direction
- **matched**: number of sources in the first plate that have a match in the second plate, after the first plate's source table gets prunned of bad detections (as per sextractor-generated parameters), and cleaned of scanner-generated artifacts (using cross-scan)
- **non-matched**: number of sources in the first plate (after it was cleaned up as above) that have **NO** match on the second plate
- **candidates**: number of candidates remaining after filtering is applyied (filtering based on Gaussian fitting, radial profile, circularity, and aperture photometry analysis), but *before* visual vetting.


**Note: this is *before* visual vetting!**

In [10]:
display_with_pandas(result)

,seq,plate 1,ut_mid_1,exptime_1,es_1,plate 2,ut_mid_2,exptime_2,es_2,total sources,matched,non-matched,candidates
0,seq01,9245,1956-09-25 19:00:47,1800,69.35,9246,1956-09-25 19:43:40,1800,68.44,590001,104546,445,1
1,seq01,9246,1956-09-25 19:43:40,1800,68.44,9247,1956-09-25 20:29:33,1800,67.58,465264,99563,299,4
2,seq03,9313,1956-12-03 17:15:31,900,62.87,9315,1956-12-03 18:06:23,900,61.75,55239,10286,30,0
3,seq03,9315,1956-12-03 18:06:23,900,61.75,9317,1956-12-03 18:56:48,900,60.63,76057,11885,237,0
4,seq03,9317,1956-12-03 18:56:48,900,60.63,9318,1956-12-03 19:26:11,900,59.99,59981,11808,48,1
5,seq03,9318,1956-12-03 19:26:11,900,59.99,9319,1956-12-03 20:27:18,900,58.8,62314,10168,43,0
6,seq03,9319,1956-12-03 20:27:18,900,58.8,9320,1956-12-03 20:55:56,900,58.28,75926,12180,51,1
7,seq04,9322,1956-12-06 18:54:56,900,61.95,9323,1956-12-06 19:19:52,900,61.4,58894,11105,47,1
8,seq04,9323,1956-12-06 19:19:52,900,61.4,9324,1956-12-06 19:44:24,900,60.91,55417,11828,53,1
9,seq04,9324,1956-12-06 19:44:24,900,60.91,9325,1956-12-06 20:08:14,900,60.41,54232,10807,36,0


In [11]:
print("Total sources:         ", np.sum(result['total sources']))
print("Total matched sources: ", np.sum(result['matched']))
print("Total non matched:     ", np.sum(result['non-matched']))
print("Total candidates:      ", np.sum(result['candidates']))

Total sources:          11691663
Total matched sources:  2495935
Total non matched:      27632
Total candidates:       311


## Final candidates tables

All candidates from all plate pairs are stored together into a single FITS table. Add some metadata to help navigate the table.

In [12]:
seq_list, table_list = map(list, zip(*candidates_list))

candidates_table = vstack(table_list)

candidates_table = add_metadata(metadata, candidates_table)

filename = os.path.join(RESULTS, "candidates_all_" + tel_suffix + ".fits")
candidates_table.write(filename, overwrite=True)
print(len(candidates_table))

311


## Visual vetting

Unfortunately, the code is not yet capable of filtering out some detections that look suspicious to the eye. In this section, we manually build the list of *source_id* values of detections that look acceptable. 

Tipically, the objects look round, have radial profiles that somewhat match star profiles, and sport cores that do not show significant asymetries.

Procedure for visual vetting:

- from the candidates table, print all sequence and source IDs;
- copy-paste the output into file *best.csv*;
- set to 1 the "remove" flag on each row that we want to ignore, editing by hand based on images and other data;
- read table from edited file *best.csv*; 
- use table's *source_id* colun to filter the candidates table;
- write result into *candidates_best.fits* file.

**Note - remember to preserve existing source IDs and sequence names**. This is the primary storage place for these IDs, which are in fact the central result of the pipeline. One can keep older versions of the *best.csv* file instead of deleting it anytime we re-run the pipeline and need to visually vet again the results.

In [13]:
# # WE DON'T NEED THIS ONCE VISUAL VETTING HAPPENS, THAT IS, ONCE THE 'best_xx.csv' FILE IS CREATED AND EDITED. 
# # But we turn it on again when analysing data from a new pipeline run.

# # we must safeguard best_xx.csv against unintentional overwrite, either by code or by manual action.
# # Every time this cell is run, it copies the existing best_xx.cv into the results/best_older directory,
# # changing its name to include a timestamp.
# timestamp = datetime.now().isoformat() 
# infile = os.path.join(RESULTS, 'best_' + tel_suffix  + '.csv')
# outfile = os.path.join(RESULTS, 'best_older', 'best_' + tel_suffix + '_' + str(timestamp) + '.csv')  
# shutil.copy(infile, outfile)

# # This creates the input to build or update file 'best_xx.csv'
# # reserve some columns for taking notes in the csv table
# print("sequence, source_id, , remove, notes1, notes2")
# for seq, table in zip(seq_list, table_list):
#     source_ids = table['source_id']
#     for sid in source_ids: 
#         print(seq + ", " + str(sid) + ", 1, ,")

Once 'best_xx.csv' is edited by hand, code below builds and writes candidates table with the "best" detections.

In [14]:
# BEWARE: this method doesn't work when a source appears more than once in the 'best_xx.csv' file.
# When creating/editing that file, use caution to look for repeats of the same plate within
# two or more sequences. Source ID is unique within the APPLAUSE context, but in this package
# we introduced the extraneous concept of 'sequence'; thus, we may have source IDs appearing
# more than once in a table, say.

table_best = Table.read(os.path.join(RESULTS, 'best_' + tel_suffix + '.csv'), format='ascii.csv')

mask = (1 - table_best['remove']).astype(bool)
table_best = table_best[mask]

mask = np.isin(candidates_table['source_id'], table_best['source_id'])  
candidates_table_best = candidates_table[mask]

filename = os.path.join(RESULTS, "candidates_best_" + tel_suffix + ".fits")
candidates_table_best.write(filename, overwrite=True)
print(len(candidates_table_best))

68


## To do

- add false positive detection from catalog (gaia) strays - DONE
- augment dataset with:
  - sequences with two plates only - DONE
  - sequences that reach over multiple nights - DONE (up to 10 nights)
- fix *footprints.ipynb*: time order in sequences of plates - DONE
- multivariate probability - NO NEED for now.
- add more shape diagnostics - DONE
- add circularity diagnostic - DONE
- profile metrics - DONE
- refactor towards batch processing - DONE
- automate data download - DONE
- add Earth shadow angle to tables - DONE